# DP2 starter notebook

Make sure you choose the **latest Weekly release** when you run this notebook on the RSP. This will give you the latest software versions with bug fixes and performance improvements.

## Setup

In [ ]:
import lsdb

import astropy.units as u
from astropy.coordinates import SkyCoord
from upath import UPath
import matplotlib.pyplot as plt

In [ ]:
# Setup
from dask.distributed import Client
client = Client(n_workers=4, memory_limit="4GiB", threads_per_worker=1)
print(f"Dask is running at: {client.dashboard_link}")

## Open catalog

In [ ]:
# Open DP2 catalog and plot
base_path = UPath("/astro/store/shire/hats/dash/hats/v30_0_6/")

cat = lsdb.open_catalog(base_path / "object_collection")
cat.plot_pixels();

In [ ]:
base_path = UPath("/astro/store/shire/hats/dash/hats/v30_0_6/")

# Choose catalog
# Object catalog: "object_collection"
# DiaObject catalog: "dia_object_collection"
cat = lsdb.open_catalog(base_path / "object_collection",
# Select columns
# TODO update column description link
# Column descriptions available at https://sdm-schemas.lsst.io/dp1.html
    columns=['g_psfFlux', 'g_psfMag', 'r_psfFlux']
# Select a sky region
    ).cone_search(ra=270.0, dec=-30.0, radius_arcsec=2*3600
# Query on column values
    ).query('g_psfFlux > 2.0 and r_psfFlux > 7.0')


In [ ]:
# Check catalog
cat

## Plot catalog

In [ ]:
# Plot catalog
cat.plot_pixels();

In [ ]:
# Plot zoomed catalog
import astropy.units as u
fov = (8 * u.deg, 8 * u.deg)
center = SkyCoord(270 * u.deg, -30 * u.deg)
fig, ax = cat.plot_pixels(projection="AIT", fov=fov, center=center);

In [ ]:
# Plot density
import hats
hats.inspection.plot_density(cat.hc_structure, ec='face', projection="AIT", fov=fov, center=center);

## Map a function

In [ ]:
# Map a function

# The mappable function takes in a dataframe that represents a partition.
# It returns a dataframe.
def g_minus_r_mapper(df):
    import numpy as np
    df['g_minus_r_mag'] = -np.log(df['g_psfFlux'] / df['r_psfFlux'])
    return df

cat = cat.map_partitions(g_minus_r_mapper)


## Compute catalog

In [ ]:
# Check catalog before computing
cat.head()

In [ ]:
# Compute catalog and write to disk
cat.write_catalog('dp2_example_cat', overwrite=True)

In [ ]:
# Compute catalog to dataframe
df = cat.compute()

In [ ]:
# Read catalog from disk
cat = lsdb.open_catalog('dp2_example_cat')

## Crossmatch

In [ ]:
# Crossmatch
gaia_cat = lsdb.open_catalog('https://data.lsdb.io/hats/gaia_dr3',
                            columns=['pmra', 'pmdec', 'parallax', 'phot_g_mean_mag'])
gaia_cat

In [ ]:
gaia_cat.plot_pixels();

In [ ]:
# Crossmatch LSST catalog with Gaia catalog
x_cat = cat.crossmatch(gaia_cat, suffix_method='overlapping_columns'
# Filter out extraneous matches: match rows should have similar magnitude
                      ).query('-1.0 < g_psfMag - phot_g_mean_mag < 1.0')

In [ ]:
x_cat.head()

In [ ]:
x_cat.plot_pixels();

In [ ]:
x_df = x_cat.compute()

In [ ]:
x_df

In [ ]:
plt.scatter(x_df['g_minus_r_mag'], x_df['parallax'], alpha=0.3);
plt.xlabel('$g-r$ (LSST)');
plt.ylabel('$\pi$ (Gaia)');

In [ ]:
## Close Dask client

In [ ]:
# Clean up
client.close()